In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# Load the dataset
dataset = pd.read_csv('medical_insurance.csv')


dataset['sex'] = dataset['sex'].map({'male': 0, 'female': 1})
dataset['smoker'] = dataset['smoker'].map({'no': 0, 'yes': 1})
dataset = pd.get_dummies(dataset, columns=['region'], drop_first=True)


In [13]:
dataset.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,1,27.900,0,1,16884.92400,False,False,True
1,18,0,33.770,1,0,1725.55230,False,True,False
2,28,0,33.000,3,0,4449.46200,False,True,False
3,33,0,22.705,0,0,21984.47061,True,False,False
4,32,0,28.880,0,0,3866.85520,True,False,False


In [14]:
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

train_labels = train_dataset.pop('charges')
test_labels = test_dataset.pop('charges')


In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_dataset = pd.DataFrame(scaler.fit_transform(train_dataset), columns=train_dataset.columns)
test_dataset = pd.DataFrame(scaler.transform(test_dataset), columns=test_dataset.columns)


In [16]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_model():
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=[train_dataset.shape[1]]),
        layers.Dense(64, activation='relu'),
        layers.Dense(1)  # output: predicted expenses
    ])

    model.compile(optimizer='adam',
                  loss='mse',
                  metrics=['mae'])
    return model

model = build_model()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
history = model.fit(
    train_dataset, train_labels,
    epochs=150,
    validation_split=0.2,
    verbose=1
)


Epoch 1/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 292791456.0000 - mae: 12759.1318 - val_loss: 318645952.0000 - val_mae: 13341.0146
Epoch 2/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 314372832.0000 - mae: 13225.8320 - val_loss: 317543456.0000 - val_mae: 13308.9316
Epoch 3/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 322920640.0000 - mae: 13339.3447 - val_loss: 314152864.0000 - val_mae: 13215.3359
Epoch 4/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 319838560.0000 - mae: 13135.3525 - val_loss: 306647488.0000 - val_mae: 13014.0146
Epoch 5/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 316953728.0000 - mae: 13005.0801 - val_loss: 293800512.0000 - val_mae: 12669.3506
Epoch 6/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 286848064.0000 - mae: 12444.5840 - val_loss: 274325024.0000 - val_mae: 12137.2422
Epoch 7/150
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 281058784.0000 - mae: 12248.5361 - val_loss: 248669616.0000 - val_mae: 11412.8779
Epoch 8/150


In [18]:
loss, mae = model.evaluate(test_dataset, test_labels, verbose=2)
print("Mean Absolute Error on test data: ${:7.2f}".format(mae))


18/18 - 0s - 3ms/step - loss: 27775260.0000 - mae: 3008.1729
Mean Absolute Error on test data: $3008.17
